# Drug Molecule Generation v2 (`PropFPToSMILES`: ZINC250K → Tox21)

## 1. Data Gathering & Configuration

Centralizes all dataset limits, model hyperparameters, and suitability thresholds in `CONFIG`, validates the CUDA environment, and downloads/caches both raw datasets:
- **ZINC250K** (`yairschiff/zinc250k` via Hugging Face `datasets`) — full dataset (`224,568` train / `24,887` validation) with `canonical_smiles`, `logP`, `qed`, and `SAS`.
- **Tox21** (`tox21.csv.gz` from DeepChem S3) — 7,831 molecules across 12 toxicity assays.

In [1]:
import argparse
import json
import math
import os
import random
import re
import sys
import time
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset as HFDataset, load_dataset
from rdkit import Chem, RDLogger
from rdkit.Chem import Crippen, QED, RDConfig, rdFingerprintGenerator
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, TensorDataset

# Silence verbose RDKit parse warnings during validity checks
RDLogger.DisableLog("rdApp.*")

# Load RDKit's built-in SA_Score module
_sa_dir = os.path.join(RDConfig.RDContribDir, "SA_Score")
if _sa_dir not in sys.path:
    sys.path.append(_sa_dir)
from rdkit.Contrib.SA_Score import sascorer

# ==============================================================================
# Centralized v2 Configuration
# ==============================================================================
CONFIG = {
    "seed": 42,
    # Dataset sizes (None = use full dataset)
    "zinc_train_limit": None,
    "zinc_val_limit": None,
    "fp_size": 2048,
    "fp_radius": 2,
    "max_len": 92,
    # Model architecture (PropFPToSMILES)
    "d_model": 256,
    "nhead": 8,
    "num_layers": 4,
    "dim_feedforward": 1024,
    "dropout": 0.1,
    "num_props": 3,  # [logP, QED, SAS]
    # ZINC250K pre-training
    "batch_size": 256,
    "zinc_epochs": 12,
    "zinc_lr": 1e-3,
    "weight_decay": 1e-2,
    # Tox21 safety classifier
    "safety_epochs": 25,
    "safety_batch_size": 256,
    "safety_lr": 1e-3,
    # Tox21 fine-tuning
    "ft_batch_size": 64,
    "ft_epochs": 10,
    "ft_lr": 2e-4,
    "ft_eta_min": 2e-5,
    # Evaluation & suitability thresholds
    "eval_gen_samples": 512,
    "epoch_gen_samples": 128,
    "logp_min": -0.4,
    "logp_max": 5.0,
    "qed_min": 0.5,
    "sas_max": 4.5,
    "non_toxic_threshold": 0.5,
}


def resolve_allow_cpu() -> bool:
    parser = argparse.ArgumentParser(add_help=False)
    parser.add_argument("--allow-cpu", action="store_true", default=False)
    args, _ = parser.parse_known_args(sys.argv[1:])
    env_flag = os.environ.get("ALLOW_CPU", "0").strip().lower() in {"1", "true", "yes"}
    return bool(args.allow_cpu or env_flag)


def validate_cuda_environment(allow_cpu: bool = False) -> torch.device:
    torch_ver = torch.__version__
    cuda_ver = torch.version.cuda
    cuda_avail = torch.cuda.is_available()

    print("=== Compute Environment Pre-Flight Check ===")
    print(f"Python executable  : {sys.executable}")
    print(f"PyTorch version    : {torch_ver}")
    print(f"CUDA build version : {cuda_ver}")
    print(f"CUDA available     : {cuda_avail}")

    if not cuda_avail:
        if not allow_cpu:
            raise RuntimeError(
                "CUDA pre-flight check failed — refusing to silently fall back to CPU. "
                "Set ALLOW_CPU=1 or pass --allow-cpu only if you intentionally want a CPU run."
            )
        torch.set_num_threads(min(12, os.cpu_count() or 4))
        return torch.device("cpu")

    dev_idx = torch.cuda.current_device()
    device = torch.device("cuda", dev_idx)
    props = torch.cuda.get_device_properties(device)
    gpu_name = torch.cuda.get_device_name(device)
    total_gib = props.total_memory / (1024 ** 3)
    print(f"GPU name           : {gpu_name}")
    print(f"Total VRAM         : {total_gib:.2f} GiB")

    smoke_a = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device=device, dtype=torch.float32)
    smoke_out = (smoke_a @ smoke_a).sum()
    torch.cuda.synchronize(device)
    print(f"CUDA smoke test    : PASSED (sum={float(smoke_out.item()):.1f} on {device})")
    return device


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


ALLOW_CPU = resolve_allow_cpu()
DEVICE = validate_cuda_environment(allow_cpu=ALLOW_CPU)
PIN_MEMORY = DEVICE.type == "cuda"
USE_AMP = DEVICE.type == "cuda"
seed_everything(CONFIG["seed"])

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
METRICS_DIR = PROJECT_ROOT / "metrics"
for d in (DATA_DIR, MODELS_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Active device      : {DEVICE} (AMP={USE_AMP}, pin_memory={PIN_MEMORY})")
print(f"Project root       : {PROJECT_ROOT}")

# Download / load raw datasets
ds = load_dataset("yairschiff/zinc250k")

tox_cache_path = DATA_DIR / "tox21.csv.gz"
tox_url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
if not tox_cache_path.exists():
    urllib.request.urlretrieve(tox_url, tox_cache_path)
tox21_raw_df = pd.read_csv(tox_cache_path, compression="gzip")

print(ds)
print(f"Tox21 shape: {tox21_raw_df.shape}")

/home/mingoll/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-zsarmojf because there was an issue with the default path (/home/mingoll/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


/home/mingoll/Projects/Mine/Molecule_Generation/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/datasets/yairschiff/zinc250k/resolve/c6e5785ff798797d818d1ae354ccbd54c10e5634/zinc250k.py


Retrying in 1s [Retry 1/5].


=== Compute Environment Pre-Flight Check ===
Python executable  : /home/mingoll/Projects/Mine/Molecule_Generation/.venv/bin/python
PyTorch version    : 2.15.0+cu132
CUDA build version : 13.2
CUDA available     : True
GPU name           : NVIDIA GeForce RTX 4050 Laptop GPU
Total VRAM         : 5.69 GiB
CUDA smoke test    : PASSED (sum=54.0 on cuda:0)
Active device      : cuda:0 (AMP=True, pin_memory=True)
Project root       : /home/mingoll/Projects/Mine/Molecule_Generation


... (8 outputs left out)

'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/datasets/yairschiff/zinc250k/resolve/c6e5785ff798797d818d1ae354ccbd54c10e5634/zinc250k.py


Using the latest cached version of the dataset since yairschiff/zinc250k couldn't be found on the Hugging Face Hub


Found the latest cached dataset configuration 'default' at /home/mingoll/.cache/huggingface/datasets/yairschiff___zinc250k/default/0.0.0/c6e5785ff798797d818d1ae354ccbd54c10e5634 (last modified on Wed Nov 25 23:47:21 2026).


DatasetDict({
    train: Dataset({
        features: ['smiles', 'logP', 'qed', 'SAS', 'canonical_smiles', 'single_bond', 'double_bond', 'triple_bond', 'aromatic_bond', 'ring_count', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R12', 'R13', 'R14', 'R15', 'R18', 'R24'],
        num_rows: 224568
    })
    validation: Dataset({
        features: ['smiles', 'logP', 'qed', 'SAS', 'canonical_smiles', 'single_bond', 'double_bond', 'triple_bond', 'aromatic_bond', 'ring_count', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R12', 'R13', 'R14', 'R15', 'R18', 'R24'],
        num_rows: 24887
    })
})
Tox21 shape: (7831, 14)


## 2. Data Preprocessing (ZINC250K)

Prepares the **full ZINC250K dataset** (controlled via `CONFIG["zinc_train_limit"]` and `CONFIG["zinc_val_limit"]`) for property-conditioned generation:
- Splits ZINC250K validation 50/50 into validation and held-out test sets.
- Builds an atom-wise SMILES tokenizer and vocabulary from the training split.
- Fits Z-score normalization (`prop_mean`, `prop_std`) on the training split's **`[logP, qed, SAS]`** columns.
- Computes 2048-bit Morgan fingerprints (stored as `uint8`) paired with normalized `[logP, qed, SAS]` vectors and padded SMILES token sequences.

In [2]:
FP_SIZE = CONFIG["fp_size"]
MAX_LEN = CONFIG["max_len"]
SEED = CONFIG["seed"]
MFP_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=CONFIG["fp_radius"], fpSize=FP_SIZE)

train_ds = ds["train"]
val_ds = ds["validation"]

if CONFIG["zinc_train_limit"] is not None:
    train_ds = train_ds.shuffle(seed=SEED).select(range(min(CONFIG["zinc_train_limit"], len(train_ds))))
if CONFIG["zinc_val_limit"] is not None:
    val_ds = val_ds.shuffle(seed=SEED).select(range(min(CONFIG["zinc_val_limit"], len(val_ds))))

val_test_split = val_ds.train_test_split(test_size=0.5, seed=SEED)
zinc_val_hf = val_test_split["train"]
zinc_test_hf = val_test_split["test"]

# Atom-wise SMILES tokenizer & vocabulary
SMILES_REGEX = re.compile(
    r"(\[[^\[\]]{1,12}\]|Br|Cl|@@|@|%[0-9]{2}|[0-9]|"
    r"[A-Za-z]|=|#|-|\+|\\|/|:|~|\.|\(|\))"
)
SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3


def tokenize_smiles(smiles: str) -> list[str]:
    return SMILES_REGEX.findall(smiles.strip())


tok_counter = Counter()
for s in train_ds["canonical_smiles"]:
    tok_counter.update(tokenize_smiles(s))

vocab_tokens = SPECIAL_TOKENS + sorted([t for t, c in tok_counter.items() if c >= 2])
token2idx = {t: i for i, t in enumerate(vocab_tokens)}
idx2token = {i: t for t, i in token2idx.items()}
print(f"Vocabulary size: {len(vocab_tokens)} tokens")

# Fit property normalization stats on ZINC250K training split: [logP, qed, SAS]
train_props_raw = np.column_stack([
    np.asarray(train_ds["logP"], dtype=np.float32),
    np.asarray(train_ds["qed"], dtype=np.float32),
    np.asarray(train_ds["SAS"], dtype=np.float32),
])
prop_mean = train_props_raw.mean(axis=0).astype(np.float32)
prop_std = np.maximum(train_props_raw.std(axis=0), 1e-6).astype(np.float32)
print(f"ZINC250K Property Mean [logP, QED, SAS]: {np.round(prop_mean, 4)}")
print(f"ZINC250K Property Std  [logP, QED, SAS]: {np.round(prop_std, 4)}")


class PropFPToSMILESDataset(Dataset):
    def __init__(
        self,
        smiles_list: list[str],
        props_raw: np.ndarray,
        token2idx: dict[str, int],
        prop_mean: np.ndarray,
        prop_std: np.ndarray,
        max_len: int = MAX_LEN,
    ):
        n = len(smiles_list)
        fps = np.empty((n, FP_SIZE), dtype=np.uint8)
        props = np.empty((n, 3), dtype=np.float32)
        tokens = np.full((n, max_len), PAD_IDX, dtype=np.int64)

        valid_count = 0
        for smi, p_raw in zip(smiles_list, props_raw):
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            fps[valid_count] = MFP_GEN.GetFingerprintAsNumPy(mol)
            props[valid_count] = (p_raw - prop_mean) / prop_std
            toks = tokenize_smiles(smi)[: max_len - 2]
            ids = [BOS_IDX] + [token2idx.get(t, UNK_IDX) for t in toks] + [EOS_IDX]
            tokens[valid_count, : len(ids)] = ids
            valid_count += 1

        self.fps = fps[:valid_count]
        self.props = props[:valid_count]
        self.tokens = tokens[:valid_count]

    def __len__(self) -> int:
        return len(self.fps)

    def __getitem__(self, idx: int):
        return (
            torch.from_numpy(self.fps[idx]).float(),
            torch.from_numpy(self.props[idx]).float(),
            torch.from_numpy(self.tokens[idx]),
        )


def _extract_zinc_props(hf_split) -> np.ndarray:
    return np.column_stack([
        np.asarray(hf_split["logP"], dtype=np.float32),
        np.asarray(hf_split["qed"], dtype=np.float32),
        np.asarray(hf_split["SAS"], dtype=np.float32),
    ])


t0 = time.time()
zinc_train_dataset = PropFPToSMILESDataset(
    train_ds["canonical_smiles"], _extract_zinc_props(train_ds), token2idx, prop_mean, prop_std
)
zinc_val_dataset = PropFPToSMILESDataset(
    zinc_val_hf["canonical_smiles"], _extract_zinc_props(zinc_val_hf), token2idx, prop_mean, prop_std
)
zinc_test_dataset = PropFPToSMILESDataset(
    zinc_test_hf["canonical_smiles"], _extract_zinc_props(zinc_test_hf), token2idx, prop_mean, prop_std
)

zinc_train_loader = DataLoader(
    zinc_train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, pin_memory=PIN_MEMORY
)
zinc_val_loader = DataLoader(
    zinc_val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, pin_memory=PIN_MEMORY
)
zinc_test_loader = DataLoader(
    zinc_test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, pin_memory=PIN_MEMORY
)

print(
    f"ZINC250K DataLoaders ready in {time.time() - t0:.1f}s -> "
    f"Train: {len(zinc_train_dataset):,} | Val: {len(zinc_val_dataset):,} | Test: {len(zinc_test_dataset):,}"
)

Vocabulary size: 57 tokens


ZINC250K Property Mean [logP, QED, SAS]: [2.4583 0.7283 3.053 ]
ZINC250K Property Std  [logP, QED, SAS]: [1.4321 0.1395 0.8342]


ZINC250K DataLoaders ready in 39.2s -> Train: 224,568 | Val: 12,443 | Test: 12,444


## 3. Train Model (via ZINC250K)

Defines `PropFPToSMILES` — an autoregressive causal Transformer decoder conditioned on two prefix tokens:
1. **Fingerprint prefix** (`2048 -> d_model`)
2. **Property prefix** (`[logP_z, qed_z, sas_z] -> d_model`)

Trains `zinc_model` on the full ZINC250K training split using `AdamW`, `OneCycleLR`, and mixed precision (`AMP`), restoring the best validation-loss checkpoint.

In [3]:
class PropFPToSMILES(nn.Module):
    def __init__(
        self,
        fp_size: int = FP_SIZE,
        num_props: int = CONFIG["num_props"],
        vocab_size: int = len(vocab_tokens),
        d_model: int = CONFIG["d_model"],
        nhead: int = CONFIG["nhead"],
        num_layers: int = CONFIG["num_layers"],
        dim_feedforward: int = CONFIG["dim_feedforward"],
        max_len: int = MAX_LEN,
        dropout: float = CONFIG["dropout"],
    ):
        super().__init__()
        self.config = {
            "fp_size": fp_size,
            "num_props": num_props,
            "vocab_size": vocab_size,
            "d_model": d_model,
            "nhead": nhead,
            "num_layers": num_layers,
            "dim_feedforward": dim_feedforward,
            "max_len": max_len,
            "dropout": dropout,
        }
        self.d_model = d_model
        self.max_len = max_len

        self.fp_proj = nn.Sequential(
            nn.Linear(fp_size, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )
        self.prop_proj = nn.Sequential(
            nn.Linear(num_props, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        self.tok_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, fp: torch.Tensor, props: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
        bsz, seq_len = input_ids.shape
        if seq_len > self.max_len:
            raise ValueError(f"Sequence length {seq_len} exceeds max_len={self.max_len}")

        fp_prefix = self.fp_proj(fp).unsqueeze(1)
        prop_prefix = self.prop_proj(props).unsqueeze(1)

        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        tok_emb = self.tok_embedding(input_ids) * math.sqrt(self.d_model) + self.pos_embedding(positions)
        tok_emb = self.emb_dropout(tok_emb)

        x = torch.cat([fp_prefix, prop_prefix, tok_emb], dim=1)
        total_len = seq_len + 2

        causal_mask = torch.triu(
            torch.ones((total_len, total_len), device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        key_padding_mask = torch.cat(
            [
                torch.zeros((bsz, 2), device=x.device, dtype=torch.bool),
                input_ids == PAD_IDX,
            ],
            dim=1,
        )

        out = self.transformer(x, mask=causal_mask, src_key_padding_mask=key_padding_mask)
        out = self.norm(out)
        return self.fc(out[:, 2:, :])

    @torch.no_grad()
    def generate(
        self,
        fp: torch.Tensor,
        props: torch.Tensor,
        idx2token: dict[int, str],
        max_len: int = MAX_LEN,
        temperature: float = 0.0,
        top_p: float = 0.9,
    ) -> list[str]:
        self.eval()
        bsz = fp.size(0)
        cur_ids = torch.full((bsz, 1), BOS_IDX, dtype=torch.long, device=fp.device)
        finished = torch.zeros(bsz, dtype=torch.bool, device=fp.device)
        decoded = [[] for _ in range(bsz)]

        for _ in range(max_len - 1):
            logits = self.forward(fp, props, cur_ids)[:, -1, :]
            logits[:, [PAD_IDX, BOS_IDX, UNK_IDX]] = float("-inf")

            if temperature <= 0.0:
                next_tok = logits.argmax(dim=-1, keepdim=True)
            else:
                scaled_logits = logits / temperature
                if 0.0 < top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True, dim=-1)
                    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_mask = cumulative_probs > top_p
                    sorted_mask[:, 1:] = sorted_mask[:, :-1].clone()
                    sorted_mask[:, 0] = False
                    sorted_logits = sorted_logits.masked_fill(sorted_mask, float("-inf"))
                    filtered_logits = torch.full_like(scaled_logits, float("-inf"))
                    filtered_logits.scatter_(1, sorted_indices, sorted_logits)
                else:
                    filtered_logits = scaled_logits
                probs = F.softmax(filtered_logits, dim=-1)
                next_tok = torch.multinomial(probs, num_samples=1)

            for i in range(bsz):
                if not finished[i]:
                    tid = int(next_tok[i, 0].item())
                    if tid == EOS_IDX:
                        finished[i] = True
                    else:
                        decoded[i].append(idx2token.get(tid, ""))

            if bool(finished.all()):
                break
            cur_ids = torch.cat([cur_ids, next_tok], dim=1)

        return ["".join(seq) for seq in decoded]


def trim_tokens(tokens: torch.Tensor) -> torch.Tensor:
    max_active = max(int((tokens != PAD_IDX).sum(dim=1).max().item()), 2)
    return tokens[:, :max_active]


@torch.no_grad()
def evaluate_val(model: nn.Module, loader: DataLoader, criterion: nn.Module, use_amp: bool = False) -> dict:
    model.eval()
    total_loss, total_correct, total_toks = 0.0, 0, 0
    for fp, props, tokens in loader:
        tokens = trim_tokens(tokens).to(DEVICE, non_blocking=True)
        fp = fp.to(DEVICE, non_blocking=True)
        props = props.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            logits = model(fp, props, tokens[:, :-1])
            targets = tokens[:, 1:]
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        mask = targets != PAD_IDX
        n_tok = int(mask.sum().item())
        preds = logits.argmax(dim=-1)
        total_correct += int((preds[mask] == targets[mask]).sum().item())
        total_loss += float(loss.item()) * n_tok
        total_toks += n_tok

    avg_loss = total_loss / max(total_toks, 1)
    return {
        "loss": avg_loss,
        "ppl": math.exp(min(avg_loss, 20.0)),
        "tok_acc": total_correct / max(total_toks, 1),
    }


@torch.no_grad()
def evaluate_generation(
    model: nn.Module,
    loader: DataLoader,
    idx2token: dict[int, str],
    n_samples: int = 512,
    temperature: float = 0.0,
    safety_clf: nn.Module | None = None,
) -> dict:
    model.eval()
    if safety_clf is not None:
        safety_clf.eval()

    collected_fp, collected_props = [], []
    total_collected = 0
    for fp_b, props_b, _ in loader:
        collected_fp.append(fp_b)
        collected_props.append(props_b)
        total_collected += fp_b.size(0)
        if total_collected >= n_samples:
            break

    fp_all = torch.cat(collected_fp, dim=0)[:n_samples]
    props_all = torch.cat(collected_props, dim=0)[:n_samples]

    gen_smiles = []
    gen_batch = 128
    for start in range(0, len(fp_all), gen_batch):
        fp_chunk = fp_all[start : start + gen_batch].to(DEVICE, non_blocking=True)
        props_chunk = props_all[start : start + gen_batch].to(DEVICE, non_blocking=True)
        gen_smiles.extend(model.generate(fp_chunk, props_chunk, idx2token, temperature=temperature))

    true_fps = fp_all.numpy() > 0.5
    valid_canon, valid_gen_fps = [], []
    precs, recs, f1s = [], [], []
    logps, qeds, sass = [], [], []
    suitable = 0

    for smi, true_fp in zip(gen_smiles, true_fps):
        mol = Chem.MolFromSmiles(smi) if smi else None
        if mol is None:
            continue
        valid_canon.append(Chem.MolToSmiles(mol, canonical=True))

        gen_fp_arr = MFP_GEN.GetFingerprintAsNumPy(mol)
        valid_gen_fps.append(gen_fp_arr)
        gen_fp = gen_fp_arr.astype(bool)

        tp = int(np.logical_and(gen_fp, true_fp).sum())
        pred_pos = int(gen_fp.sum())
        true_pos = int(true_fp.sum())
        p = tp / pred_pos if pred_pos > 0 else 0.0
        r = tp / true_pos if true_pos > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        precs.append(p)
        recs.append(r)
        f1s.append(f1)

        logp = float(Crippen.MolLogP(mol))
        qed = float(QED.qed(mol))
        sas = float(sascorer.calculateScore(mol))
        logps.append(logp)
        qeds.append(qed)
        sass.append(sas)

        if (
            CONFIG["logp_min"] <= logp <= CONFIG["logp_max"]
            and qed >= CONFIG["qed_min"]
            and sas <= CONFIG["sas_max"]
        ):
            suitable += 1

    n_valid = len(valid_canon)
    non_toxic_prob = None
    if safety_clf is not None and n_valid > 0:
        fp_tensor = torch.from_numpy(np.stack(valid_gen_fps)).float().to(DEVICE)
        toxic_probs = torch.sigmoid(safety_clf(fp_tensor)).cpu().numpy()
        non_toxic_prob = float(np.mean(1.0 - toxic_probs))

    return {
        "validity": n_valid / max(len(gen_smiles), 1),
        "uniqueness": len(set(valid_canon)) / max(n_valid, 1) if n_valid > 0 else 0.0,
        "precision": float(np.mean(precs)) if precs else 0.0,
        "recall": float(np.mean(recs)) if recs else 0.0,
        "f1": float(np.mean(f1s)) if f1s else 0.0,
        "logP": float(np.mean(logps)) if logps else 0.0,
        "QED": float(np.mean(qeds)) if qeds else 0.0,
        "SAS": float(np.mean(sass)) if sass else 0.0,
        "suitable": suitable / max(n_valid, 1) if n_valid > 0 else 0.0,
        "sample": valid_canon[0] if valid_canon else (gen_smiles[0] if gen_smiles else "<empty>"),
        "non_toxic_prob": non_toxic_prob,
    }


def print_eval_box(title: str, metrics: dict) -> None:
    nt_str = f"{metrics['non_toxic_prob']:.4f}" if metrics.get("non_toxic_prob") is not None else "N/A (pre-Tox21)"
    print(f"================ {title} ================")
    print(f"  1. Validity       : {metrics['validity']:.4f}")
    print(f"  2. Uniqueness     : {metrics['uniqueness']:.4f}")
    print(f"  3. Precision      : {metrics['precision']:.4f}")
    print(f"  4. Recall         : {metrics['recall']:.4f}")
    print(f"  5. F1             : {metrics['f1']:.4f}")
    print(f"  6. LogP           : {metrics['logP']:.4f}")
    print(f"  7. QED            : {metrics['QED']:.4f}")
    print(f"  8. SAS            : {metrics['SAS']:.4f}")
    print(f"  9. Suitable       : {metrics['suitable']:.4f}")
    print(f" 10. Sample         : {metrics['sample']}")
    print(f" 11. Non-Toxic Prob : {nt_str}")
    print("=" * (34 + len(title)))


# Instantiate & train base ZINC250K model
zinc_model = PropFPToSMILES().to(DEVICE)
print(f"PropFPToSMILES parameters: {sum(p.numel() for p in zinc_model.parameters()):,} | Device: {DEVICE} | AMP: {USE_AMP}")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.AdamW(zinc_model.parameters(), lr=CONFIG["zinc_lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=CONFIG["zinc_lr"],
    epochs=CONFIG["zinc_epochs"],
    steps_per_epoch=len(zinc_train_loader),
    pct_start=0.1,
)
scaler = torch.amp.GradScaler("cuda" if USE_AMP else "cpu", enabled=USE_AMP)

best_val_loss = float("inf")
best_zinc_weights = None
zinc_history = []

for epoch in range(1, CONFIG["zinc_epochs"] + 1):
    t0 = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)

    zinc_model.train()
    train_loss_sum, train_correct, train_toks = 0.0, 0, 0

    for fp, props, tokens in zinc_train_loader:
        tokens = trim_tokens(tokens).to(DEVICE, non_blocking=True)
        fp = fp.to(DEVICE, non_blocking=True)
        props = props.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = zinc_model(fp, props, tokens[:, :-1])
            targets = tokens[:, 1:]
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(zinc_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        with torch.no_grad():
            mask = targets != PAD_IDX
            n_tok = int(mask.sum().item())
            train_correct += int((logits.argmax(dim=-1)[mask] == targets[mask]).sum().item())
            train_loss_sum += float(loss.item()) * n_tok
            train_toks += n_tok

    train_loss = train_loss_sum / max(train_toks, 1)
    train_acc = train_correct / max(train_toks, 1)
    lr = optimizer.param_groups[0]["lr"]

    val = evaluate_val(zinc_model, zinc_val_loader, criterion, use_amp=USE_AMP)
    gen = evaluate_generation(
        zinc_model, zinc_val_loader, idx2token, n_samples=CONFIG["epoch_gen_samples"], temperature=0.0
    )

    is_best = val["loss"] < best_val_loss
    if is_best:
        best_val_loss = val["loss"]
        best_zinc_weights = {k: v.detach().cpu().clone() for k, v in zinc_model.state_dict().items()}

    vram_str = ""
    if DEVICE.type == "cuda":
        peak_gib = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3)
        total_gib = torch.cuda.get_device_properties(DEVICE).total_memory / (1024 ** 3)
        vram_str = f" | Peak VRAM: {peak_gib:.2f}/{total_gib:.2f} GiB"

    zinc_history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val["loss"], "val_tok_acc": val["tok_acc"], **gen})
    best_tag = " [BEST]" if is_best else ""
    print(
        f"[ZINC250K v2] Epoch {epoch:02d}/{CONFIG['zinc_epochs']:02d}{best_tag} | "
        f"LR: {lr:.2e} | Time: {time.time() - t0:.1f}s{vram_str}\n"
        f"  Train -> Loss: {train_loss:.4f} | PPL: {math.exp(min(train_loss, 20.0)):.2f} | TokAcc: {train_acc:.4f}\n"
        f"  Val   -> Loss: {val['loss']:.4f} | PPL: {val['ppl']:.2f} | TokAcc: {val['tok_acc']:.4f}\n"
        f"  Gen   -> Valid: {gen['validity']:.4f} | Unique: {gen['uniqueness']:.4f} | "
        f"Prec: {gen['precision']:.4f} | Rec: {gen['recall']:.4f} | F1: {gen['f1']:.4f} | "
        f"logP: {gen['logP']:.2f} | QED: {gen['QED']:.3f} | SAS: {gen['SAS']:.2f} | Suit: {gen['suitable']:.2%}"
    )

zinc_model.load_state_dict(best_zinc_weights)

PropFPToSMILES parameters: 3,738,937 | Device: cuda:0 | AMP: True


[ZINC250K v2] Epoch 01/12 [BEST] | LR: 9.36e-04 | Time: 59.5s | Peak VRAM: 1.04/5.69 GiB
  Train -> Loss: 1.0724 | PPL: 2.92 | TokAcc: 0.6532
  Val   -> Loss: 0.4704 | PPL: 1.60 | TokAcc: 0.8288
  Gen   -> Valid: 0.3438 | Unique: 1.0000 | Prec: 0.5762 | Rec: 0.5049 | F1: 0.5333 | logP: 1.98 | QED: 0.728 | SAS: 2.89 | Suit: 84.09%


[ZINC250K v2] Epoch 02/12 [BEST] | LR: 9.86e-04 | Time: 59.4s | Peak VRAM: 1.04/5.69 GiB
  Train -> Loss: 0.4247 | PPL: 1.53 | TokAcc: 0.8453
  Val   -> Loss: 0.2716 | PPL: 1.31 | TokAcc: 0.8989
  Gen   -> Valid: 0.6172 | Unique: 1.0000 | Prec: 0.6929 | Rec: 0.6673 | F1: 0.6783 | logP: 2.33 | QED: 0.749 | SAS: 2.98 | Suit: 84.81%


[ZINC250K v2] Epoch 03/12 [BEST] | LR: 9.33e-04 | Time: 59.4s | Peak VRAM: 1.04/5.69 GiB
  Train -> Loss: 0.2928 | PPL: 1.34 | TokAcc: 0.8920
  Val   -> Loss: 0.2001 | PPL: 1.22 | TokAcc: 0.9255
  Gen   -> Valid: 0.6641 | Unique: 1.0000 | Prec: 0.7783 | Rec: 0.7558 | F1: 0.7656 | logP: 2.03 | QED: 0.741 | SAS: 3.03 | Suit: 78.82%


[ZINC250K v2] Epoch 04/12 [BEST] | LR: 8.43e-04 | Time: 59.2s | Peak VRAM: 1.04/5.69 GiB
  Train -> Loss: 0.2342 | PPL: 1.26 | TokAcc: 0.9128
  Val   -> Loss: 0.1684 | PPL: 1.18 | TokAcc: 0.9364
  Gen   -> Valid: 0.7969 | Unique: 1.0000 | Prec: 0.8075 | Rec: 0.7938 | F1: 0.7994 | logP: 2.38 | QED: 0.754 | SAS: 3.01 | Suit: 84.31%


[ZINC250K v2] Epoch 05/12 [BEST] | LR: 7.24e-04 | Time: 59.2s | Peak VRAM: 1.04/5.69 GiB
  Train -> Loss: 0.1984 | PPL: 1.22 | TokAcc: 0.9253
  Val   -> Loss: 0.1510 | PPL: 1.16 | TokAcc: 0.9420
  Gen   -> Valid: 0.7188 | Unique: 1.0000 | Prec: 0.8488 | Rec: 0.8386 | F1: 0.8429 | logP: 2.14 | QED: 0.757 | SAS: 3.08 | Suit: 82.61%


... (4 outputs left out)

[ZINC250K v2] Epoch 10/12 [BEST] | LR: 8.22e-05 | Time: 59.5s | Peak VRAM: 1.04/5.69 GiB
  Train -> Loss: 0.1094 | PPL: 1.12 | TokAcc: 0.9567
  Val   -> Loss: 0.0951 | PPL: 1.10 | TokAcc: 0.9627
  Gen   -> Valid: 0.8516 | Unique: 1.0000 | Prec: 0.9044 | Rec: 0.9000 | F1: 0.9018 | logP: 2.36 | QED: 0.752 | SAS: 3.09 | Suit: 80.73%


[ZINC250K v2] Epoch 11/12 [BEST] | LR: 2.10e-05 | Time: 59.3s | Peak VRAM: 1.04/5.69 GiB
  Train -> Loss: 0.1021 | PPL: 1.11 | TokAcc: 0.9592
  Val   -> Loss: 0.0922 | PPL: 1.10 | TokAcc: 0.9639
  Gen   -> Valid: 0.8984 | Unique: 1.0000 | Prec: 0.9083 | Rec: 0.9027 | F1: 0.9050 | logP: 2.45 | QED: 0.748 | SAS: 3.05 | Suit: 80.00%


[ZINC250K v2] Epoch 12/12 [BEST] | LR: 4.03e-09 | Time: 59.2s | Peak VRAM: 1.04/5.69 GiB
  Train -> Loss: 0.0986 | PPL: 1.10 | TokAcc: 0.9604
  Val   -> Loss: 0.0916 | PPL: 1.10 | TokAcc: 0.9641
  Gen   -> Valid: 0.8906 | Unique: 1.0000 | Prec: 0.9097 | Rec: 0.9011 | F1: 0.9049 | logP: 2.47 | QED: 0.750 | SAS: 3.07 | Suit: 81.58%


<All keys matched successfully>

## 4. Evaluate Model (`zinc_model` on ZINC250K Held-Out Test Data)

Evaluates the trained base `zinc_model` on `zinc_test_loader` and prints the standardized 11-metric evaluation box.

In [4]:
zinc_test_lm = evaluate_val(zinc_model, zinc_test_loader, criterion, use_amp=USE_AMP)
zinc_eval_metrics = {
    **zinc_test_lm,
    **evaluate_generation(
        zinc_model,
        zinc_test_loader,
        idx2token,
        n_samples=CONFIG["eval_gen_samples"],
        temperature=0.0,
        safety_clf=None,
    ),
}
print(f"ZINC250K Test LM -> Loss: {zinc_test_lm['loss']:.4f} | PPL: {zinc_test_lm['ppl']:.2f} | TokAcc: {zinc_test_lm['tok_acc']:.4f}")
print_eval_box("ZINC250K Base Model (v2) Test Evaluation", zinc_eval_metrics)

ZINC250K Test LM -> Loss: 0.0912 | PPL: 1.10 | TokAcc: 0.9638
================ ZINC250K Base Model (v2) Test Evaluation ================
  1. Validity       : 0.9199
  2. Uniqueness     : 1.0000
  3. Precision      : 0.8967
  4. Recall         : 0.8895
  5. F1             : 0.8926
  6. LogP           : 2.4590
  7. QED            : 0.7289
  8. SAS            : 3.0351
  9. Suitable       : 0.8301
 10. Sample         : CSc1ccc([C@@H](O)[C@@H](C)c2ccccn2)cc1
 11. Non-Toxic Prob : N/A (pre-Tox21)


## 5. Extract Model as a File

Saves `zinc_model` along with its architecture config, SMILES vocabulary, property normalization stats (`prop_mean`, `prop_std`), `CONFIG`, and evaluation metrics to `models/zinc250k_prop_fp2smiles_v2.pt`, then runs a round-trip load verification.

In [5]:
ZINC_MODEL_PATH = MODELS_DIR / "zinc250k_prop_fp2smiles_v2.pt"


def save_model_file(
    path: Path,
    model: PropFPToSMILES,
    version_name: str,
    metrics: dict,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "version": version_name,
            "model_config": model.config,
            "config": CONFIG,
            "token2idx": token2idx,
            "idx2token": idx2token,
            "prop_mean": prop_mean,
            "prop_std": prop_std,
            "metrics": metrics,
            "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        },
        path,
    )


def load_model_file(path: Path, device: torch.device = DEVICE) -> tuple[PropFPToSMILES, dict]:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m = PropFPToSMILES(**ckpt["model_config"]).to(device)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    return m, ckpt


save_model_file(ZINC_MODEL_PATH, zinc_model, "v2.0-zinc250k-prop-fp2smiles", zinc_eval_metrics)
print(f"Saved base model -> {ZINC_MODEL_PATH} ({ZINC_MODEL_PATH.stat().st_size / (1024 * 1024):.2f} MB)")

_test_m, _test_ckpt = load_model_file(ZINC_MODEL_PATH)
assert _test_ckpt["version"] == "v2.0-zinc250k-prop-fp2smiles"
del _test_m, _test_ckpt
print("Round-trip load check passed.")

Saved base model -> /home/mingoll/Projects/Mine/Molecule_Generation/models/zinc250k_prop_fp2smiles_v2.pt (14.29 MB)
Round-trip load check passed.


## 6. Data Preprocessing (Tox21)

Aggregates all 12 Tox21 toxicity assays into a single binary label regardless of toxicity type:
- `toxic_sum = sum(tested_assays)` → **`is_toxic = 1`** if `toxic_sum >= 1`, else **`0`** (non-toxic).

Two outputs are produced:
1. **Binary Toxicity Classifier (`Tox21BinarySafetyClassifier`)**: Trained on 2048-bit Morgan fingerprints across all labeled Tox21 molecules (`models/tox21_safety_classifier_v2.pt`) to score `Non-Toxic Prob = 1 - P(toxic)` for any generated molecule.
2. **Non-Toxic Fine-Tuning DataLoaders**: Strictly non-toxic (`is_toxic == 0`), salt-stripped, vocabulary-compatible Tox21 molecules with RDKit-computed `[logP, QED, SAS]` normalized using ZINC250K stats (80/10/10 Train/Val/Test).

In [6]:
TOX21_ASSAYS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase", "NR-ER", "NR-ER-LBD",
    "NR-PPAR-gamma", "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

# Part A: Build binary toxic/non-toxic dataset for Tox21BinarySafetyClassifier
clf_fps, clf_labels = [], []
non_toxic_smiles, non_toxic_props = [], []
seen_all, seen_nontox = set(), set()

for _, row in tox21_raw_df.iterrows():
    if pd.isna(row["smiles"]):
        continue
    smi = str(row["smiles"]).strip()
    if not smi:
        continue
    if "." in smi:
        smi = max(smi.split("."), key=len)

    assay_vals = row[TOX21_ASSAYS].dropna().values.astype(int)
    if len(assay_vals) == 0:
        continue

    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() < 3:
        continue
    can_smi = Chem.MolToSmiles(mol, canonical=True)

    # Collapse all 12 toxicity columns: toxic if sum >= 1, else non-toxic (0)
    is_toxic = 1 if int(assay_vals.sum()) >= 1 else 0

    if can_smi not in seen_all:
        seen_all.add(can_smi)
        clf_fps.append(MFP_GEN.GetFingerprintAsNumPy(mol))
        clf_labels.append(is_toxic)

    # Part B: Keep strictly non-toxic molecules compatible with ZINC250K vocabulary
    if is_toxic == 0 and can_smi not in seen_nontox:
        toks = tokenize_smiles(can_smi)
        if 4 <= len(toks) <= MAX_LEN - 2 and all(t in token2idx for t in toks):
            seen_nontox.add(can_smi)
            non_toxic_smiles.append(can_smi)
            non_toxic_props.append([
                float(Crippen.MolLogP(mol)),
                float(QED.qed(mol)),
                float(sascorer.calculateScore(mol)),
            ])

X_clf = np.asarray(clf_fps, dtype=np.float32)
y_clf = np.asarray(clf_labels, dtype=np.float32)
print(
    f"Tox21 Binary Dataset: {len(y_clf):,} unique molecules "
    f"(Non-Toxic: {int((y_clf == 0).sum()):,} | Toxic: {int((y_clf == 1).sum()):,})"
)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_clf, y_clf, test_size=0.20, stratify=y_clf, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

clf_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
    batch_size=CONFIG["safety_batch_size"],
    shuffle=True,
)
clf_val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va)),
    batch_size=CONFIG["safety_batch_size"],
    shuffle=False,
)
clf_test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te)),
    batch_size=CONFIG["safety_batch_size"],
    shuffle=False,
)


class Tox21BinarySafetyClassifier(nn.Module):
    def __init__(self, fp_size: int = FP_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(fp_size, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


tox_safety_clf = Tox21BinarySafetyClassifier(FP_SIZE).to(DEVICE)
pos_weight = torch.tensor([(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)], dtype=torch.float32, device=DEVICE)
clf_crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
clf_opt = torch.optim.AdamW(tox_safety_clf.parameters(), lr=CONFIG["safety_lr"], weight_decay=1e-2)

best_auc, best_clf_weights = -1.0, None
for ep in range(1, CONFIG["safety_epochs"] + 1):
    tox_safety_clf.train()
    for xb, yb in clf_train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        clf_opt.zero_grad(set_to_none=True)
        loss = clf_crit(tox_safety_clf(xb), yb)
        loss.backward()
        clf_opt.step()

    tox_safety_clf.eval()
    va_probs, va_true = [], []
    with torch.no_grad():
        for xb, yb in clf_val_loader:
            va_probs.append(torch.sigmoid(tox_safety_clf(xb.to(DEVICE))).cpu().numpy())
            va_true.append(yb.numpy())
    va_auc = roc_auc_score(np.concatenate(va_true), np.concatenate(va_probs))
    if va_auc > best_auc:
        best_auc = va_auc
        best_clf_weights = {k: v.detach().cpu().clone() for k, v in tox_safety_clf.state_dict().items()}

tox_safety_clf.load_state_dict(best_clf_weights)
tox_safety_clf.eval()
with torch.no_grad():
    te_probs = np.concatenate([torch.sigmoid(tox_safety_clf(xb.to(DEVICE))).cpu().numpy() for xb, _ in clf_test_loader])
te_auc = roc_auc_score(y_te, te_probs)
te_f1 = f1_score(y_te, (te_probs >= 0.5).astype(int))
SAFETY_CLF_PATH = MODELS_DIR / "tox21_safety_classifier_v2.pt"
torch.save({"state_dict": best_clf_weights, "test_auc": te_auc, "test_f1": te_f1}, SAFETY_CLF_PATH)
print(f"Tox21BinarySafetyClassifier -> Val AUC: {best_auc:.4f} | Test AUC: {te_auc:.4f} | Test Toxic F1: {te_f1:.4f}")
print(f"Saved safety classifier -> {SAFETY_CLF_PATH}")

# Part B: Build Non-Toxic Fine-Tuning DataLoaders (80/10/10)
non_toxic_props = np.asarray(non_toxic_props, dtype=np.float32)
idx_all = np.arange(len(non_toxic_smiles))
idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.20, random_state=SEED)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=SEED)

tox_train_ds = PropFPToSMILESDataset(
    [non_toxic_smiles[i] for i in idx_tr], non_toxic_props[idx_tr], token2idx, prop_mean, prop_std
)
tox_val_ds = PropFPToSMILESDataset(
    [non_toxic_smiles[i] for i in idx_va], non_toxic_props[idx_va], token2idx, prop_mean, prop_std
)
tox_test_ds = PropFPToSMILESDataset(
    [non_toxic_smiles[i] for i in idx_te], non_toxic_props[idx_te], token2idx, prop_mean, prop_std
)

tox_train_loader = DataLoader(tox_train_ds, batch_size=CONFIG["ft_batch_size"], shuffle=True, pin_memory=PIN_MEMORY)
tox_val_loader = DataLoader(tox_val_ds, batch_size=CONFIG["ft_batch_size"], shuffle=False, pin_memory=PIN_MEMORY)
tox_test_loader = DataLoader(tox_test_ds, batch_size=CONFIG["ft_batch_size"], shuffle=False, pin_memory=PIN_MEMORY)

print(
    f"Tox21 Non-Toxic Splits -> Train: {len(tox_train_ds):,} | "
    f"Val: {len(tox_val_ds):,} | Test: {len(tox_test_ds):,}"
)

Tox21 Binary Dataset: 7,677 unique molecules (Non-Toxic: 4,861 | Toxic: 2,816)


Tox21BinarySafetyClassifier -> Val AUC: 0.7559 | Test AUC: 0.7662 | Test Toxic F1: 0.6024
Saved safety classifier -> /home/mingoll/Projects/Mine/Molecule_Generation/models/tox21_safety_classifier_v2.pt


Tox21 Non-Toxic Splits -> Train: 3,759 | Val: 470 | Test: 470


## 7. Fine-Tuning (`zinc_model` on Tox21 Non-Toxic Data)

Loads the pre-trained ZINC250K v2 checkpoint (`models/zinc250k_prop_fp2smiles_v2.pt`) into `tox_model`, logs its zero-shot baseline on `tox_val_loader`, and fine-tunes exclusively on verified **non-toxic** Tox21 molecules conditioned on `[logP, QED, SAS]`. Restores the best validation weights and saves `models/tox21_prop_fp2smiles_v2.pt`.

In [7]:
tox_model, tox_ckpt = load_model_file(ZINC_MODEL_PATH)
tox_idx2token = tox_ckpt["idx2token"]

ft_optimizer = torch.optim.AdamW(tox_model.parameters(), lr=CONFIG["ft_lr"], weight_decay=CONFIG["weight_decay"])
ft_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer, T_max=CONFIG["ft_epochs"], eta_min=CONFIG["ft_eta_min"]
)
ft_scaler = torch.amp.GradScaler("cuda" if USE_AMP else "cpu", enabled=USE_AMP)

init_val = evaluate_val(tox_model, tox_val_loader, criterion, use_amp=USE_AMP)
best_tox_val = init_val["loss"]
best_tox_weights = {k: v.detach().cpu().clone() for k, v in tox_model.state_dict().items()}
print(
    f"Initial Tox21 Val (before fine-tuning) -> "
    f"Loss: {init_val['loss']:.4f} | PPL: {init_val['ppl']:.2f} | TokAcc: {init_val['tok_acc']:.4f}"
)

tox_history = []
for epoch in range(1, CONFIG["ft_epochs"] + 1):
    t0 = time.time()
    tox_model.train()
    train_loss_sum, train_correct, train_toks = 0.0, 0, 0

    for fp, props, tokens in tox_train_loader:
        tokens = trim_tokens(tokens).to(DEVICE, non_blocking=True)
        fp = fp.to(DEVICE, non_blocking=True)
        props = props.to(DEVICE, non_blocking=True)

        ft_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = tox_model(fp, props, tokens[:, :-1])
            targets = tokens[:, 1:]
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        ft_scaler.scale(loss).backward()
        ft_scaler.unscale_(ft_optimizer)
        nn.utils.clip_grad_norm_(tox_model.parameters(), 1.0)
        ft_scaler.step(ft_optimizer)
        ft_scaler.update()

        with torch.no_grad():
            mask = targets != PAD_IDX
            n_tok = int(mask.sum().item())
            train_correct += int((logits.argmax(dim=-1)[mask] == targets[mask]).sum().item())
            train_loss_sum += float(loss.item()) * n_tok
            train_toks += n_tok

    ft_scheduler.step()
    train_loss = train_loss_sum / max(train_toks, 1)
    train_acc = train_correct / max(train_toks, 1)
    lr = ft_optimizer.param_groups[0]["lr"]

    val = evaluate_val(tox_model, tox_val_loader, criterion, use_amp=USE_AMP)
    gen = evaluate_generation(
        tox_model, tox_val_loader, tox_idx2token, n_samples=64, temperature=0.0, safety_clf=tox_safety_clf
    )

    is_best = val["loss"] < best_tox_val
    if is_best:
        best_tox_val = val["loss"]
        best_tox_weights = {k: v.detach().cpu().clone() for k, v in tox_model.state_dict().items()}

    tox_history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val["loss"], **gen})
    best_tag = " [BEST]" if is_best else ""
    print(
        f"[Tox21 Fine-Tune v2] Epoch {epoch:02d}/{CONFIG['ft_epochs']:02d}{best_tag} | "
        f"LR: {lr:.2e} | Time: {time.time() - t0:.1f}s\n"
        f"  Train -> Loss: {train_loss:.4f} | PPL: {math.exp(min(train_loss, 20.0)):.2f} | TokAcc: {train_acc:.4f}\n"
        f"  Val   -> Loss: {val['loss']:.4f} | PPL: {val['ppl']:.2f} | TokAcc: {val['tok_acc']:.4f}\n"
        f"  Gen   -> Valid: {gen['validity']:.4f} | F1: {gen['f1']:.4f} | "
        f"logP: {gen['logP']:.2f} | QED: {gen['QED']:.3f} | SAS: {gen['SAS']:.2f} | "
        f"Suit: {gen['suitable']:.2%} | Non-Toxic Prob: {gen['non_toxic_prob']:.4f}"
    )

tox_model.load_state_dict(best_tox_weights)
TOX_MODEL_PATH = MODELS_DIR / "tox21_prop_fp2smiles_v2.pt"
save_model_file(TOX_MODEL_PATH, tox_model, "v2.0-tox21-prop-fp2smiles", tox_history[-1])
print(f"Saved fine-tuned model -> {TOX_MODEL_PATH} ({TOX_MODEL_PATH.stat().st_size / (1024 * 1024):.2f} MB)")

Initial Tox21 Val (before fine-tuning) -> Loss: 0.6044 | PPL: 1.83 | TokAcc: 0.8475


[Tox21 Fine-Tune v2] Epoch 01/10 [BEST] | LR: 1.96e-04 | Time: 1.5s
  Train -> Loss: 0.4563 | PPL: 1.58 | TokAcc: 0.8511
  Val   -> Loss: 0.3315 | PPL: 1.39 | TokAcc: 0.8863
  Gen   -> Valid: 0.8125 | F1: 0.7522 | logP: 1.61 | QED: 0.560 | SAS: 2.34 | Suit: 55.77% | Non-Toxic Prob: 0.7691


[Tox21 Fine-Tune v2] Epoch 02/10 [BEST] | LR: 1.83e-04 | Time: 1.7s
  Train -> Loss: 0.3346 | PPL: 1.40 | TokAcc: 0.8839
  Val   -> Loss: 0.3096 | PPL: 1.36 | TokAcc: 0.8965
  Gen   -> Valid: 0.8438 | F1: 0.7880 | logP: 1.50 | QED: 0.556 | SAS: 2.45 | Suit: 55.56% | Non-Toxic Prob: 0.7572


[Tox21 Fine-Tune v2] Epoch 03/10 [BEST] | LR: 1.63e-04 | Time: 1.5s
  Train -> Loss: 0.2848 | PPL: 1.33 | TokAcc: 0.8988
  Val   -> Loss: 0.2943 | PPL: 1.34 | TokAcc: 0.9009
  Gen   -> Valid: 0.8281 | F1: 0.8084 | logP: 1.74 | QED: 0.530 | SAS: 2.46 | Suit: 47.17% | Non-Toxic Prob: 0.7637


[Tox21 Fine-Tune v2] Epoch 04/10 [BEST] | LR: 1.38e-04 | Time: 1.7s
  Train -> Loss: 0.2506 | PPL: 1.28 | TokAcc: 0.9106
  Val   -> Loss: 0.2874 | PPL: 1.33 | TokAcc: 0.9058
  Gen   -> Valid: 0.8281 | F1: 0.7944 | logP: 1.77 | QED: 0.545 | SAS: 2.49 | Suit: 49.06% | Non-Toxic Prob: 0.7572


[Tox21 Fine-Tune v2] Epoch 05/10 [BEST] | LR: 1.10e-04 | Time: 1.7s
  Train -> Loss: 0.2229 | PPL: 1.25 | TokAcc: 0.9192
  Val   -> Loss: 0.2772 | PPL: 1.32 | TokAcc: 0.9093
  Gen   -> Valid: 0.8750 | F1: 0.8202 | logP: 1.85 | QED: 0.538 | SAS: 2.50 | Suit: 50.00% | Non-Toxic Prob: 0.7582


[Tox21 Fine-Tune v2] Epoch 06/10 | LR: 8.22e-05 | Time: 1.7s
  Train -> Loss: 0.2014 | PPL: 1.22 | TokAcc: 0.9269
  Val   -> Loss: 0.2776 | PPL: 1.32 | TokAcc: 0.9104
  Gen   -> Valid: 0.7812 | F1: 0.8343 | logP: 1.55 | QED: 0.541 | SAS: 2.42 | Suit: 52.00% | Non-Toxic Prob: 0.7885


[Tox21 Fine-Tune v2] Epoch 07/10 [BEST] | LR: 5.71e-05 | Time: 1.5s
  Train -> Loss: 0.1843 | PPL: 1.20 | TokAcc: 0.9333
  Val   -> Loss: 0.2771 | PPL: 1.32 | TokAcc: 0.9114
  Gen   -> Valid: 0.8281 | F1: 0.8134 | logP: 1.45 | QED: 0.554 | SAS: 2.54 | Suit: 54.72% | Non-Toxic Prob: 0.7841


[Tox21 Fine-Tune v2] Epoch 08/10 [BEST] | LR: 3.72e-05 | Time: 1.5s
  Train -> Loss: 0.1729 | PPL: 1.19 | TokAcc: 0.9372
  Val   -> Loss: 0.2752 | PPL: 1.32 | TokAcc: 0.9124
  Gen   -> Valid: 0.8438 | F1: 0.8416 | logP: 1.64 | QED: 0.539 | SAS: 2.45 | Suit: 50.00% | Non-Toxic Prob: 0.7721


[Tox21 Fine-Tune v2] Epoch 09/10 | LR: 2.44e-05 | Time: 1.7s
  Train -> Loss: 0.1655 | PPL: 1.18 | TokAcc: 0.9397
  Val   -> Loss: 0.2755 | PPL: 1.32 | TokAcc: 0.9142
  Gen   -> Valid: 0.8594 | F1: 0.8554 | logP: 1.70 | QED: 0.544 | SAS: 2.46 | Suit: 52.73% | Non-Toxic Prob: 0.7661


[Tox21 Fine-Tune v2] Epoch 10/10 | LR: 2.00e-05 | Time: 1.5s
  Train -> Loss: 0.1609 | PPL: 1.17 | TokAcc: 0.9415
  Val   -> Loss: 0.2769 | PPL: 1.32 | TokAcc: 0.9127
  Gen   -> Valid: 0.8438 | F1: 0.8513 | logP: 1.79 | QED: 0.545 | SAS: 2.43 | Suit: 55.56% | Non-Toxic Prob: 0.7698
Saved fine-tuned model -> /home/mingoll/Projects/Mine/Molecule_Generation/models/tox21_prop_fp2smiles_v2.pt (14.29 MB)


## 8. Evaluate Model (`tox_model` on Tox21 Held-Out Test Data)

Evaluates the fine-tuned `tox_model` across the entire held-out `tox_test_loader` and prints all 11 required metrics including `Non-Toxic Prob`.

In [8]:
tox_test_lm = evaluate_val(tox_model, tox_test_loader, criterion, use_amp=USE_AMP)
tox_eval_metrics = {
    **tox_test_lm,
    **evaluate_generation(
        tox_model,
        tox_test_loader,
        tox_idx2token,
        n_samples=len(tox_test_loader.dataset),
        temperature=0.0,
        safety_clf=tox_safety_clf,
    ),
}
print(f"Tox21 Test LM -> Loss: {tox_test_lm['loss']:.4f} | PPL: {tox_test_lm['ppl']:.2f} | TokAcc: {tox_test_lm['tok_acc']:.4f}")
print_eval_box("Tox21 Fine-Tuned Model (v2) Test Evaluation", tox_eval_metrics)

Tox21 Test LM -> Loss: 0.2801 | PPL: 1.32 | TokAcc: 0.9096
================ Tox21 Fine-Tuned Model (v2) Test Evaluation ================
  1. Validity       : 0.8936
  2. Uniqueness     : 0.9929
  3. Precision      : 0.8425
  4. Recall         : 0.8297
  5. F1             : 0.8329
  6. LogP           : 2.2044
  7. QED            : 0.5527
  8. SAS            : 2.5333
  9. Suitable       : 0.5738
 10. Sample         : CC(C)CCCCCCOC(=O)c1ccccc1C(=O)OCCCCCCC(C)C
 11. Non-Toxic Prob : 0.7488


## 9. Calculate F1 & Version Comparison

Compares the extracted **Base ZINC250K v2** checkpoint against the **Fine-Tuned Tox21 v2** checkpoint on the exact same held-out `tox_test_loader`, computing both **Fingerprint F1** and **Safe-Drug F1** (harmonic mean of Valid × Suitable × Non-Toxic precision and Fingerprint Recall). Saves results to `metrics/v2_f1_leaderboard.json` and `metrics/v2_f1_comparison.png`.

In [9]:
base_v2_model, base_v2_ckpt = load_model_file(ZINC_MODEL_PATH)
ft_v2_model, ft_v2_ckpt = load_model_file(TOX_MODEL_PATH)

base_on_tox = evaluate_generation(
    base_v2_model,
    tox_test_loader,
    base_v2_ckpt["idx2token"],
    n_samples=len(tox_test_loader.dataset),
    temperature=0.0,
    safety_clf=tox_safety_clf,
)
ft_on_tox = tox_eval_metrics


def calc_safe_drug_f1(m: dict) -> float:
    safe_prec = m["validity"] * m["suitable"] * (m["non_toxic_prob"] or 0.0)
    rec = m["recall"]
    return (2.0 * safe_prec * rec / (safe_prec + rec)) if (safe_prec + rec) > 0 else 0.0


base_on_tox["safe_drug_f1"] = calc_safe_drug_f1(base_on_tox)
ft_on_tox["safe_drug_f1"] = calc_safe_drug_f1(ft_on_tox)

print_eval_box("Base ZINC250K v2 on Tox21 Test", base_on_tox)
print_eval_box("Fine-Tuned Tox21 v2 on Tox21 Test", ft_on_tox)

comparison_df = pd.DataFrame([
    {"Model": "v2.0-zinc250k-base", **{k: base_on_tox[k] for k in ["validity", "uniqueness", "precision", "recall", "f1", "logP", "QED", "SAS", "suitable", "non_toxic_prob", "safe_drug_f1"]}},
    {"Model": "v2.0-tox21-finetuned", **{k: ft_on_tox[k] for k in ["validity", "uniqueness", "precision", "recall", "f1", "logP", "QED", "SAS", "suitable", "non_toxic_prob", "safe_drug_f1"]}},
])
print("\n=== Final F1 & Safety Leaderboard (Tox21 Test Set) ===")
print(comparison_df.to_string(index=False))

leaderboard_path = METRICS_DIR / "v2_f1_leaderboard.json"
leaderboard_path.write_text(
    json.dumps({"base_v2": base_on_tox, "finetuned_v2": ft_on_tox}, indent=2),
    encoding="utf-8",
)
print(f"Saved leaderboard -> {leaderboard_path}")

# Visual comparison chart
plot_metrics = ["validity", "precision", "recall", "f1", "suitable", "non_toxic_prob", "safe_drug_f1"]
plot_labels = ["Validity", "Precision", "Recall", "FP F1", "Suitable", "Non-Toxic Prob", "Safe-Drug F1"]
x = np.arange(len(plot_metrics))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(x - width / 2, [base_on_tox[m] for m in plot_metrics], width, label="ZINC250K Base (v2)")
ax.bar(x + width / 2, [ft_on_tox[m] for m in plot_metrics], width, label="Tox21 Fine-Tuned (v2)")
ax.set_ylabel("Score")
ax.set_title("Molecule Generation v2: Base vs. Tox21 Fine-Tuned (Held-Out Tox21 Test)")
ax.set_xticks(x)
ax.set_xticklabels(plot_labels, rotation=15)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend()
plt.tight_layout()

chart_path = METRICS_DIR / "v2_f1_comparison.png"
plt.savefig(chart_path, dpi=150)
plt.close(fig)
print(f"Saved comparison chart -> {chart_path}")

================ Base ZINC250K v2 on Tox21 Test ================
  1. Validity       : 0.8702
  2. Uniqueness     : 0.9902
  3. Precision      : 0.7627
  4. Recall         : 0.7543
  5. F1             : 0.7539
  6. LogP           : 2.0683
  7. QED            : 0.5643
  8. SAS            : 2.5602
  9. Suitable       : 0.6015
 10. Sample         : CC(C)CCCCOC(=O)c1ccccc1-c1ccccc1C(C)C
 11. Non-Toxic Prob : 0.7323
================ Fine-Tuned Tox21 v2 on Tox21 Test ================
  1. Validity       : 0.8936
  2. Uniqueness     : 0.9929
  3. Precision      : 0.8425
  4. Recall         : 0.8297
  5. F1             : 0.8329
  6. LogP           : 2.2044
  7. QED            : 0.5527
  8. SAS            : 2.5333
  9. Suitable       : 0.5738
 10. Sample         : CC(C)CCCCCCOC(=O)c1ccccc1C(=O)OCCCCCCC(C)C
 11. Non-Toxic Prob : 0.7488

=== Final F1 & Safety Leaderboard (Tox21 Test Set) ===
               Model  validity  uniqueness  precision   recall       f1     logP      QED      SAS  suitab